# Kaggle Homework 4
## 1. Feature Engineering & Multi-Model Ensembling

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

## 2. Load Data and Engineer New Features
We will explore feature interactions (Temperature and Humidity), transformations, and grouping aggregations (average rainfall by region).

In [ ]:
# Load data
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')
sample_sub = pd.read_csv('../data/sample_submission.csv')

test_ids = test['id'].copy()
train = train.drop(columns=['id'])
test = test.drop(columns=['id'])

# Map Target
ordinal_map = {'Low': 0, 'Medium': 1, 'High': 2}
train['Irrigation_Need'] = train['Irrigation_Need'].map(ordinal_map)

y = train['Irrigation_Need']
X = train.drop(columns=['Irrigation_Need'])

# Combine for consistent feature engineering
X_all = pd.concat([X, test], axis=0).reset_index(drop=True)

###########################################
# Feature Engineering
###########################################
# 1. Interaction Feature: Heat Index proxy (Temp * Humidity)
X_all['Temp_Humidity_Index'] = X_all['Temperature_C'] * (X_all['Humidity'] / 100.0)

# 2. Ratio Feature: Aridity Proxy (Temp / Rainfall)
X_all['Aridity_Proxy'] = X_all['Temperature_C'] / (X_all['Rainfall_mm'] + 1.0)

# 3. Grouping: Average Rainfall by Crop Type
crop_rainfall_mean = X_all.groupby('Crop_Type')['Rainfall_mm'].transform('mean')
X_all['Crop_Mean_Rainfall_Diff'] = X_all['Rainfall_mm'] - crop_rainfall_mean
###########################################

# Split back
X_engineered = X_all.iloc[:len(train)].copy()
test_engineered = X_all.iloc[len(train):].copy()

X_train, X_val, y_train, y_val = train_test_split(X_engineered, y, test_size=0.2, random_state=42, stratify=y)

## 3. Preprocessing

In [ ]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", MinMaxScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

def to_dense(x):
    return x.toarray() if hasattr(x, "toarray") else x
dense_transformer = FunctionTransformer(to_dense, accept_sparse=True)

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

## 4. Build Multiple Meaningfully Different Models
We use LightGBM, CatBoost, and Gaussian Naive Bayes to ensure diverse model types for ensembling.

In [ ]:
# Model 1: LightGBM (Leaf-wise Gradient Boosting)
lgbm = Pipeline([
    ('prep', preprocessor),
    ('model', LGBMClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, objective="multiclass", random_state=42, verbose=-1))
])

# Model 2: CatBoost (Symmetric Trees, specifically good with categoricals, but we use preprocessed data here for consistency)
catb = Pipeline([
    ('prep', preprocessor),
    ('model', CatBoostClassifier(iterations=100, depth=6, learning_rate=0.1, loss_function='MultiClass', verbose=0, random_state=42))
])

# Model 3: Gaussian Naive Bayes (Probabilistic, heavily differs from tree ensembles)
nb = Pipeline([
    ('prep', preprocessor),
    ('to_dense', dense_transformer),
    ('model', GaussianNB())
])

# Fit base models (ignoring sample weights for NB pipeline directly, standard fit)
print("Training LightGBM...")
lgbm.fit(X_train, y_train, model__sample_weight=sample_weights)

print("Training CatBoost...")
catb.fit(X_train, y_train)

print("Training Naive Bayes...")
nb.fit(X_train, y_train)

print("Training complete.")

## 5. Evaluate Feature Importance
Using LightGBM's built-in feature importance to see if our engineered features matter.

In [ ]:
lgbm_model = lgbm.named_steps['model']
feature_names = lgbm.named_steps['prep'].get_feature_names_out()

importances = lgbm_model.feature_importances_
imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
imp_df = imp_df.sort_values(by='Importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
plt.barh(imp_df['Feature'], imp_df['Importance'])
plt.gca().invert_yaxis()
plt.title('Top 15 Feature Importances (LightGBM)')
plt.xlabel('Importance')
plt.show()

print("Engineered features like Temp_Humidity_Index or Crop_Mean_Rainfall_Diff will appear here if the model found them structurally useful for splitting nodes.")

## 6. Ensemble: Probability Averaging vs Stacking

In [ ]:
# Soft Voting (Probability Averaging)
voting_clf = VotingClassifier(
    estimators=[('lgbm', lgbm), ('catb', catb), ('nb', nb)],
    voting='soft',
    weights=[2, 2, 1] # Slightly lower weight for NB as tree models typically dominate
)

# Stacking (Meta-Model)
stacking_clf = StackingClassifier(
    estimators=[('lgbm', lgbm), ('catb', catb), ('nb', nb)],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=3,
    n_jobs=-1
)

print("Training Voting Ensemble...")
voting_clf.fit(X_train, y_train)

print("Training Stacking Ensemble (this takes a moment due to CV)...")
stacking_clf.fit(X_train, y_train)

print("Ensemble Training Complete.")

## 7. Model Evaluation & Comparison

In [ ]:
models = {
    'LightGBM': lgbm,
    'CatBoost': catb,
    'Naive Bayes': nb,
    'Voting Ensemble': voting_clf,
    'Stacking Ensemble': stacking_clf
}

results = []
for name, model in models.items():
    preds = model.predict(X_val)
    acc = accuracy_score(y_val, preds)
    bal_acc = balanced_accuracy_score(y_val, preds)
    results.append({'Model': name, 'Accuracy': acc, 'Balanced Accuracy': bal_acc})

results_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False)
print(results_df.to_string(index=False))

## 8. Generate Final Submission
We use the best ensemble (usually Stacking or Voting) to generate the final predictions.

In [ ]:
# Assuming Stacking is competitive, we will use it for the final submission
final_preds = stacking_clf.predict(test_engineered)

reverse_map = {0: 'Low', 1: 'Medium', 2: 'High'}
final_preds_labels = [reverse_map[p] for p in final_preds]

sub = pd.DataFrame({'id': test_ids, 'Irrigation_Need': final_preds_labels})
sub.to_csv('submission_hw4.csv', index=False)
print("Saved submission_hw4.csv successfully!")
sub.head()